# Introduction to Deep Learning, Assignment 2, Task 2


# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2025-12-18 15:52:48.450268: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-18 15:52:48.478775: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-18 15:52:54.867457: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


## Model builder functions

### Pre training model

In [6]:
# Defining the calculator datasets. From X_img to X_text_onehot
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [7]:
# Your code is: damn code

from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate,GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2, L1L2


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength):
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image


    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(X_in)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    #residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8), name = "residual_branch"))(B1_final)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=128, 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            #recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder

In [8]:
def build_image2text_pretraining(dropout = 0.5, max_size=512,RLstrength=1.0e-4):

    vocab_size = 15

    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(6, vocab_size))

    encoder = build_image2text_encoder(dropout,RLstrength)
    _, hidden, cell = encoder(X_in)
    
    h_flattened = GlobalAveragePooling2D(name='h_flattened')(hidden)#Flatten()(hidden)
    h_initial = Dense(max_size, kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = GlobalAveragePooling2D(name='c_flattened')(cell)#Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]
    
    output, hidden, cell = LSTM(
        max_size, 
        return_sequences = True,
        return_state=True, 
        dropout = dropout, 
        #recurrent_dropout = dropout, 
        name='lstm_pre_training',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(Y_in, initial_state = ini_state)
        
    dense = TimeDistributed(Dense(vocab_size, activation='softmax', name = 'decoder_dense_pre_training'))
    y_out = dense(output)

    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'pretraining_model')
#    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
#    full.compile(
#        loss=loss, optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full



### Calculator model

In [9]:
# Defining the calculator datasets. From X_text_onehot to y_text_onehot
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [12]:
# Here we build the full model
def build_text2text_calc(dropout = 0.5, max_size=512, learning_rate = 2.5e-4, max_answer_length_tf=4,RLstrength=1.0e-4):

    vocab_size = len(vocabulary_tf)

    # Define input layer of full model
    X_in = Input(shape = (6, vocab_size), name = 'expression_input')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)), name = "answer")


    # calculator encoder
    encoder_lstm = LSTM(max_size, return_state=True, return_sequences=True, name = 'calculator_encoder')
    key, hidden, cell = encoder_lstm(X_in)
    ini_state = [hidden, cell]    


    decoder_lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        ) #
    

    query, _, _ = decoder_lstm(Y_in, initial_state = ini_state)

    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = TimeDistributed(Activation('relu'), name = "decoder_activation")(final)
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    
    y_out = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'calculator')
#    full.compile(
#        loss='categorical_crossentropy', optimizer=AdamW(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full

### Concatenate visual encoder and calculator

In [ ]:
# Defining the full datasets. From X_img to y_text_onehot
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_grafted_model(vision_model, calc_model):
    """
    Implements the grafting plan:
    1. Vision: ConvLSTM2D output (Sequence & States).
    2. Interface: Project Vision features to match Calculator dimensions.
    3. Calculator: Decoder LSTM + Attention using Vision features.
    """
    
    # --- 1. INPUTS ---
    image_input = vision_model.input[0]          # (None, 5, 28, 28, 1)
    answer_input = calc_model.get_layer('answer').input # (None, 4, 15) for Teacher Forcing

    # --- 2. VISION EXTRACTION (The Graft) ---
    # We grab the output of the ConvLSTM layer directly
    # vision_model.get_layer('encoder_model').output returns [Sequence, h, c]
    conv_out = vision_model.get_layer('encoder_model').output
    
    raw_seq = conv_out[0] # (None, 5, 7, 7, 128) - The "Video"
    raw_h   = conv_out[1] # (None, 7, 7, 128)   - The "State"
    raw_c   = conv_out[2] # (None, 7, 7, 128)   - The "Memory"

    # --- 3. DIMENSION PROJECTION (The Interface) ---
    # The Calculator expects 256 units, Vision gives 128. We must project.
    
    # A. Prepare Sequence for Attention (Collapse spatial dims -> Project)
    # (None, 5, 7, 7, 128) -> (None, 5, 128) -> (None, 5, 256)
    seq_pooled = layers.TimeDistributed(layers.GlobalAveragePooling2D())(raw_seq)
    seq_proj   = layers.TimeDistributed(layers.Dense(256, activation='tanh'))(seq_pooled)
    
    # B. Prepare States for Initialization (Collapse spatial dims -> Project)
    h_proj = layers.Dense(256, activation='tanh')(layers.GlobalAveragePooling2D()(raw_h))
    c_proj = layers.Dense(256, activation='tanh')(layers.GlobalAveragePooling2D()(raw_c))

    # --- 4. CALCULATOR DECODER (The Brain) ---
    # We initialize the Decoder LSTM with the projected Vision States
    decoder_lstm = calc_model.get_layer('decoder_lstm')
    
    # Run the LSTM on the 'answer_input' (Teacher Forcing)
    # It returns the hidden states at each step
    decoder_seq, _, _ = decoder_lstm(answer_input, initial_state=[h_proj, c_proj])

    # --- 5. ATTENTION MECHANISM (The Eyes) ---
    # The Attention block now looks at the PROJECTED VISION SEQUENCE
    attention_block = calc_model.get_layer('attention_block')
    
    # Query: Decoder Output (Where am I in the calculation?)
    # Value: Vision Sequence (What do the digits look like?)
    context_vector = attention_block([decoder_seq, seq_proj])

    # --- 6. RECONSTRUCTION (The Output) ---
    # Re-wire the rest of the calculator head
    concat_layer = calc_model.get_layer('concat_q_A')
    merged = concat_layer([decoder_seq, context_vector])
    
    combined_dense = calc_model.get_layer('combined_dense')
    combined = combined_dense(merged)
    
    # Residual Connection Handling
    res_dense = calc_model.get_layer('decoder_res_dense')
    res_path = res_dense(answer_input)
    
    add_layer = calc_model.get_layer('decoder_with_residual')
    residual = add_layer([combined, res_path])
    
    # Final Norm & Softmax
    norm = calc_model.get_layer('decoder_layer_norm')
    act = calc_model.get_layer('decoder_activation')
    final_dense = calc_model.get_layer('decoder_dense')
    
    output = final_dense(norm(act(residual)))

    # --- 7. COMPILE ---
    model = models.Model(inputs=[image_input, answer_input], outputs=output)
    
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4),
        metrics=['categorical_accuracy']
    )
    
    return model

## Training the models

In [10]:
# Training the visual encoder. We train this beforehand.
## Building model
dropout=0.5
RLstrength=0
max_size=256
image2text_pretraining = build_image2text_pretraining(dropout=dropout, RLstrength=RLstrength, max_size=max_size)


## Compile
learning_rate=4.0e-4 # Initial LR
weight_decay=3.0e-4  # Decoupled Weight Decay

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


## Training
stopper_patience_warmup = 10

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks = [early_stopper])

### Recompile so AdamW moments are reset
image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


scheduler_patience = 5
stopper_patience = 20

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper],
               verbose=0)


I0000 00:00:1765999332.639538 2479847 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6127 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_1  │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed    │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_1  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_2  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_3  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_4  │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_6  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_7  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_8  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_9  │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │      2,112 │ -                 │
│ time_distributed_5  │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

...

Epoch 1/50


E0000 00:00:1765999336.513308 2479847 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_340/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_340/gra

500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - categorical_accuracy: 0.4280 - loss: 1.8127 - val_categorical_accuracy: 0.4455 - val_loss: 1.8223
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - categorical_accuracy: 0.6459 - loss: 1.3581 - val_categorical_accuracy: 0.5587 - val_loss: 1.5891
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - categorical_accuracy: 0.8102 - loss: 1.0505 - val_categorical_accuracy: 0.6440 - val_loss: 1.4163
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.8887 - loss: 0.8823 - val_categorical_accuracy: 0.7503 - val_loss: 1.1734
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - categorical_accuracy: 0.9356 - loss: 0.7778 - val_categorical_accuracy: 0.7733 - val_loss: 1.1707
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9555 - loss: 0.7230 - val_categorical_accuracy: 0.8028 - val_loss: 1.1084
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9650 - 

E0000 00:00:1766000285.486540 2479847 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_5', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid_2' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/


Epoch 18: ReduceLROnPlateau reducing learning rate to 0.00019999999494757503.

Epoch 23: ReduceLROnPlateau reducing learning rate to 9.999999747378752e-05.

Epoch 28: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 34: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.

Epoch 39: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.

Epoch 44: ReduceLROnPlateau reducing learning rate to 6.24999984211172e-06.

Epoch 49: ReduceLROnPlateau reducing learning rate to 3.12499992105586e-06.


In [13]:
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
max_size=256
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size)

## Compile
learning_rate=5.0e-4 
weight_decay=1.0e-4  

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

## Training
stopper_patience_warmup = 10

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history_warmup = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 50,
               batch_size = 64,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks = [early_stopper])

### Recompile so AdamW momenta are reset
text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

scheduler_patience = 3
stopper_patience = 12

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 120,
               batch_size = 64,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper],
               verbose=0)

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ answer[0][0],     │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - categorical_accuracy: 0.5251 - loss: 2.0136 - val_categorical_accuracy: 0.5570 - val_loss: 1.8285
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.5704 - loss: 1.7657 - val_categorical_accuracy: 0.5853 - val_loss: 1.6854
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - categorical_accuracy: 0.5958 - loss: 1.6404 - val_categorical_accuracy: 0.6192 - val_loss: 1.5706
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - categorical_accuracy: 0.6315 - loss: 1.5237 - val_categorical_accuracy: 0.6585 - val_loss: 1.4627
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - categorical_accuracy: 0.6592 - loss: 1.4418 - val_categorical_accuracy: 0.6739 - val_loss: 1.3913
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - categorical_accuracy: 0.6793 - loss: 1.3816 - val_categorical_accuracy: 0.6857 - val_loss: 1.3411
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - categorical_accuracy: 0.693

In [ ]:
# Building full model
image2text = build_image2text(image2text_pretraining, text2text_calculator)

# Phase 1: Static visual encoder weights
image2text_pretraining.trainable = False

learning_rate_static = 5e-4
weight_decay_static = 1e-4

optimizer_static = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_static,
    weight_decay=weight_decay_static
)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text.compile(
    optimizer=optimizer_static, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

image2text.summary(expand_nested=False)

history_full_static = image2text.fit(
    x=[X_train, y_train_in], 
    y=y_train_target, 
    epochs=7,
    batch_size=32,
    validation_data=([X_val, y_val_in], y_val_target)
)

# Phase 2: Dynamic visual encoder weights
image2text_pretraining.trainable = True

learning_rate_dynamic = 1.0e-5
weight_decay_dynamic = 5e-4 

optimizer_dynamic = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_dynamic,
    weight_decay=weight_decay_dynamic
)

image2text.compile(
    optimizer=optimizer_dynamic, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

stopper_patience = 25
scheduler_patience = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

history_full_dynamic = image2text.fit(
    x=[X_train, y_train_in], 
    y=y_train_target, 
    epochs=60,
    batch_size=32,
    validation_data=([X_val, y_val_in], y_val_target),
    callbacks=[lr_scheduler, early_stopper]
)

Model: "full_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ answer_teacher_for… │ (None, 4, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 15)     │          0 │ answer_teacher_f… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_sequence      │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 6, 15)     │          0 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pretraining_model   │ (None, 6, 15)     │  1,254,991 │ image_sequence[0… │
│ (Functional)        │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator          │ (None, 4, 15)     │    696,591 │ pretraining_mode… │
│ (Functional)        │                   │            │ answer_teacher_f… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,951,582 (7.44 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 1,254,991 (4.79 MB)

Epoch 1/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - categorical_accuracy: 0.9354 - loss: 0.7857 - val_categorical_accuracy: 0.9042 - val_loss: 0.9398
Epoch 2/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9890 - loss: 0.6528 - val_categorical_accuracy: 0.9103 - val_loss: 0.9269
Epoch 3/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9911 - loss: 0.6436 - val_categorical_accuracy: 0.9130 - val_loss: 0.9200
Epoch 4/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9928 - loss: 0.6369 - val_categorical_accuracy: 0.9122 - val_loss: 0.9129
Epoch 5/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - categorical_accuracy: 0.9930 - loss: 0.6319 - val_categorical_accuracy: 0.9110 - val_loss: 0.9215
Epoch 6/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - categorical_accuracy: 0.9937 - loss: 0.6286 - val_categorical_accuracy: 0.9099 - val_loss: 0.9171
Epoch 7/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - categorical_accuracy: 0.993

E0000 00:00:1765984485.613792 2298204 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_127/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_127/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while_grad/body/_943/gradient_tape/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/gradients/full_model_1/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/full_model_1/pretr

250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 73ms/step - categorical_accuracy: 0.9977 - loss: 0.6115 - val_categorical_accuracy: 0.9172 - val_loss: 0.8996 - learning_rate: 1.0000e-05
Epoch 2/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9983 - loss: 0.6066 - val_categorical_accuracy: 0.9128 - val_loss: 0.9059 - learning_rate: 1.0000e-05
Epoch 3/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9981 - loss: 0.6054 - val_categorical_accuracy: 0.9152 - val_loss: 0.9030 - learning_rate: 1.0000e-05
Epoch 4/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9985 - loss: 0.6036 - val_categorical_accuracy: 0.9145 - val_loss: 0.9022 - learning_rate: 1.0000e-05
Epoch 5/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9983 - loss: 0.6033 - val_categorical_accuracy: 0.9155 - val_loss: 0.8987 - learning_rate: 1.0000e-05
Epoch 6/120
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - categorical_accuracy: 0.9984 - loss: 0.6

## Inference

In [ ]:
# Defining the inference procedure

def predict_math_expression(image_sequence, model, index_to_char, 
                            start_token='<start>', end_token='<end>', 
                            max_len=4, vocab_size=15):

    # Prepare Inputs
    img_input = np.expand_dims(image_sequence, axis=0)
    
    # Initialize decoder input with zeros
    decoder_input = np.zeros((1, max_len, vocab_size))
    
    # Find the index for start token
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    # Seed the first position with <start>
    decoder_input[0, 0, start_idx] = 1.0
    
    predicted_indices = [start_idx]
    
    # Recursive Loop
    for i in range(max_len - 1):
        # Predict the next token
        preds = model.predict([img_input, decoder_input], verbose=0)
        
        # Get the character at the current step (i+1)
        next_idx = np.argmax(preds[0, i, :])
        predicted_indices.append(next_idx)
        
        # Stop if we hit the end token
        if next_idx == end_idx:
            break
            
        # Update the buffer for the next iteration
        if i + 1 < max_len:
            decoder_input[0, i + 1, next_idx] = 1.0

    # Map to String
    result_str = "".join([index_to_char[idx] for idx in predicted_indices 
                          if index_to_char[idx] not in [start_token, end_token]])
    
    return result_str

In [31]:
# Evaluate 10 random input images
start = random.randint(0,1000)
stop = start+10
sample_imgs = X_test[start: stop]
a = decode_labels_tf(y_test[start:stop])
expressions = []
for img in sample_imgs:
    expression = predict_math_expression(img, image2text, reverse_indices)
    expressions.append(expression)

print(a, expressions)

['80 ', '156', '94 ', '-78', '-18', '116', '43 ', '78 ', '109', '143'] ['70 ', '156', '84 ', '-78', '-18', '116', '45 ', '78 ', '109', '70 ']


#### Batch inference or something

In [40]:
# Batch inference. This is faster.
def evaluate_calculator_batch(test_images, test_targets, vision_model, calc_model, 
                              index_to_char, start_token='<start>', end_token='<end>', 
                              max_len=4, vocab_size=15):

    num_samples = test_images.shape[0]
    
    # Visual encoder
    zero_array = np.zeros((num_samples, 6, vocab_size))
    expression_probs = vision_model.predict([test_images, zero_array], batch_size=32, verbose=1)
    
    # Calculator Phase: Recursive decoding in a batch loop
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    decoded_answer = np.zeros((num_samples, max_len, vocab_size))
    decoded_answer[:, 0, start_idx] = 1.0
    
    # Track which samples in the batch have hit the end token
    finished = np.zeros(num_samples, dtype=bool)
    final_indices = np.full((num_samples, max_len), end_idx)
    final_indices[:, 0] = start_idx

    for i in range(max_len - 1):
        # We only run the calculator model here (very fast)
        preds = calc_model.predict([expression_probs, decoded_answer], batch_size=num_samples, verbose=0)
        
        # Look at the prediction for the NEXT character
        next_indices = np.argmax(preds[:, i, :], axis=-1)
        
        for b_idx in range(num_samples):
            if not finished[b_idx]:
                token = next_indices[b_idx]
                final_indices[b_idx, i+1] = token
                decoded_answer[b_idx, i+1, token] = 1.0
                if token == end_idx:
                    finished[b_idx] = True
        
        if np.all(finished): break

    # Calculation of Metrics
    true_indices = np.argmax(test_targets, axis=-1)
    
    token_correct = 0
    math_correct = 0
    
    for i in range(num_samples):
        p_str = "".join([index_to_char[idx] for idx in final_indices[i] if index_to_char[idx] not in [start_token, end_token, ' ']])
        t_str = "".join([index_to_char[idx] for idx in true_indices[i] if index_to_char[idx] not in [start_token, end_token, ' ']])
        
        # Math Accuracy
        if p_str == t_str:
            math_correct += 1
            
        # Token Accuracy
        for t_step in range(max_len):
            if final_indices[i, t_step] == true_indices[i, t_step]:
                token_correct += 1
                
    total_tokens = num_samples * max_len
    
    print(f"\nFinal Math Correctness: {math_correct/num_samples:.2%}")
    print(f"Final Token Accuracy: {token_correct/total_tokens:.2%}")
    
    return math_correct/num_samples, token_correct/total_tokens

In [41]:
# Use the fast batch evaluator

X_sample_set = X_test
y_sample_set = y_test

math_acc, token_acc = evaluate_calculator_batch(
    X_sample_set, y_sample_set, 
    image2text_pretraining, text2text_calculator, 
    reverse_indices
)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Final Math Correctness: 78.00%
Final Token Accuracy: 90.91%


In [27]:
image2text.save(f'math_acc_{math_acc:3f}.keras')

In [ ]:
import numpy as np

def inference_vision_batch(vision_model, image_sequences, batch_size=32):
    """
    Applies vectorized inference to transform images into expression probabilities.
    image_sequences: Shape (N, 5, 28, 28, 1)
    """
    num_samples = image_sequences.shape[0]
    
    # We provide a dummy input for the teacher-forcing branch (input_2)
    # Since the Vision model's output doesn't rely on this during inference
    dummy_input = np.zeros((num_samples, 6, 15)) 
    
    # Predict in batches to avoid OOM (Out of Memory) errors
    expression_probs = vision_model.predict(
        [image_sequences, dummy_input], 
        batch_size=batch_size, 
        verbose=0
    )
    
    return expression_probs

In [42]:
image2text_pretraining.summary(expand_nested=True, line_length=120)

print("\n--- CALCULATOR SUMMARY ---")
text2text_calculator.summary(expand_nested=True, line_length=120)

Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ sequence (InputLayer)             │ (None, 5, 28, 28, 1)         │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ encoder_model (Functional)        │ [(None, 5, 7, 7, 128),       │           906,560 │ sequence[0][0]            
│                                   │ (None, 7, 7, 128), (None, 7, │                   │                           
│                                   │ 7, 128)]                     │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ input_layer_5 (InputLayer)   │ (None, 5, 28, 28, 1)         │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_26          │ (None, 5, 28, 28, 32)        │               320 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_27          │ (None, 5, 28, 28, 32)        │               128 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_28          │ (None, 5, 28, 28, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_29          │ (None, 5, 28, 28, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_30          │ (None, 5, 14, 14, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_32          │ (None, 5, 14, 14, 64)        │            18,496 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_33          │ (None, 5, 14, 14, 64)        │               256 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_34          │ (None, 5, 14, 14, 64)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼───────────────────

 Total params: 3,764,591 (14.36 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ expression_input (InputLayer)     │ (None, 6, 15)                │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ answer (InputLayer)               │ (None, 4, 15)                │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ calculator_encoder (LSTM)         │ [(None, 6, 256), (None,      │           278,528 │ expression_input[0][0]    
│                                   │ 256), (None, 256)]           │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_lstm (LSTM)               │ [(None, 4, 256), (None,      │           278,528 │ answer[0][0],             
│                                   │ 256), (None, 256)]           │                   │ calculator_encoder[0][1], 
│                                   │                              │                   │ calculator_encoder[0][2]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ attention_block (Attention)       │ (None, 4, 256)               │                 0 │ decoder_lstm[0][0],       
│                                   │                              │                   │ calculator_encoder[0][0]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ concat_q_A (Concatenate)          │ (None, 4, 512)               │                 0 │ decoder_lstm[0][0],       
│                                   │                              │                   │ attention_block[0][0]     
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ combined_dense (Dense)            │ (None, 4, 256)               │           131,328 │ concat_q_A[0][0]          
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_res_dense                 │ (None, 4, 256)               │             3,840 │ answer[0][0]              
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_with_residual (Add)       │ (None, 4, 256)               │                 0 │ combined_dense[0][0],     
│                                   │                              │                   │ decoder_res_dense[0][0]   
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_activation                │ (None, 4, 256)               │                 0 │ decoder_with_residual[0][0
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_layer_norm                │ (None, 4, 256)               │               512 │ decoder_activation[0][0]  
│ (LayerNormalization)              │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_dense (TimeDistributed)   │ (None, 4, 15)     

 Total params: 2,089,775 (7.97 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,393,184 (5.31 MB)